### 1.
Load the movies.dat and ratings.dat files from the ml-10m dataset into two DataFrames.Merge the two DataFrames and create a column named "year"
that contains the year in which each rating was given.


In [1]:
import sys
import os

# Add the project root to the path so we can import 'src'
# Assuming our notebook is in 'recommender-ml10m/notebooks/'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data_loader import DataLoader

# Initialize with a path relative to the PROJECT ROOT
# This 'data' folder will be created in our main project directory
loader = DataLoader(base_path=os.path.join(project_root, 'data'))

# Download, extract, and load the dataset
df = loader.get_processed_data()

print(f"Data shape: {df.shape}")
print(df.head())

Merging data...
Converting timestamps...
Data shape: (10000054, 6)
   userId  movieId  rating                 title  \
0       1      122     5.0      Boomerang (1992)   
1       1      185     5.0       Net, The (1995)   
2       1      231     5.0  Dumb & Dumber (1994)   
3       1      292     5.0       Outbreak (1995)   
4       1      316     5.0       Stargate (1994)   

                         genres  year  
0                Comedy|Romance  1996  
1         Action|Crime|Thriller  1996  
2                        Comedy  1996  
3  Action|Drama|Sci-Fi|Thriller  1996  
4       Action|Adventure|Sci-Fi  1996  


### 2.
Find the top 5 genres that experienced the greatest decrease in average annual rating from the first year data is available to the last.
Note that the earliest year with ratings may differ per genre.


In [2]:
from src.analyzer import GenreAnalyzer

analyzer = GenreAnalyzer(df)
summary = analyzer.analyze_trends()

# --- View 1: The Raw Exploded Data (Massive Table) ---
# Be careful with .head(), this table has ~25 million rows!
print("Exploded View Sample:")
display(analyzer.exploded_df.head(15))

# --- View 2: The Yearly Averages for EVERY genre/year combination ---
print("\nAnnual Stats Table:")
display(analyzer.annual_stats.sort_values(['genre', 'year']))

# --- View 3: The Full Results (Not just top 5) ---
print("\nTop 5 Genres with Greatest Decrease in avg annual rating:")
display(analyzer.summary_df.head(5))

# --- Insight for Top 5 Genres with Greatest Decrease  ---
# Look for genres with a small 'start_count'.
# These are the ones where the 'earliest year' likely skewed the results.

Exploding genres...
Exploded View Sample:


,year,rating,genres,genre
0,1996,5.0,Comedy|Romance,Comedy
0,1996,5.0,Comedy|Romance,Romance
1,1996,5.0,Action|Crime|Thriller,Action
1,1996,5.0,Action|Crime|Thriller,Crime
1,1996,5.0,Action|Crime|Thriller,Thriller
2,1996,5.0,Comedy,Comedy
3,1996,5.0,Action|Drama|Sci-Fi|Thriller,Action
3,1996,5.0,Action|Drama|Sci-Fi|Thriller,Drama
3,1996,5.0,Action|Drama|Sci-Fi|Thriller,Sci-Fi
3,1996,5.0,Action|Drama|Sci-Fi|Thriller,Thriller



Annual Stats Table:


,genre,year,avg_rating,rating_count
0,(no genres listed),2004,2.750000,2
1,(no genres listed),2007,4.000000,3
2,(no genres listed),2008,4.000000,2
3,Action,1995,3.000000,1
4,Action,1996,3.440509,331387
...,...,...,...,...
271,Western,2005,3.460447,23753
272,Western,2006,3.525736,14940
273,Western,2007,3.529675,11845
274,Western,2008,3.598341,12660



Top 5 Genres with Greatest Decrease in avg annual rating:


,genre,start_year,end_year,start_rating,end_rating,start_count,decrease
11,Horror,1995,2009,5.000000,3.310457,1,1.689543
17,Thriller,1995,2009,5.000000,3.472470,1,1.527530
14,Mystery,1995,2009,5.000000,3.659091,1,1.340909
6,Crime,1995,2009,4.000000,3.619338,2,0.380662
18,War,1996,2009,3.943008,3.723529,59973,0.219478


### 3.
 Is the previous analysis regarding the decline in genre popularity affected by the number of
ratings? Would you take this into account? Make an adjustment for this and comment on the results.

####
    The decline in genre popularity is absolutely affected by the number of ratings. The year a genre starts appearing in the dataset plays a massive role
    for three main reasons:
    1. Selection Bias (The "Early Adopter" Effect): In 1995–1997, users of MovieLens were primarily tech-savvy early adopters, researchers, or hardcore cinephiles.
       Their rating behavior differs significantly from the "general public" who joined in 2005.
    2. Sample Size Sensitivity: If the "Film-Noir" genre has its first rating in 1996 with only 10 total ratings, a single 1-star review will tank the average.
       By 2009, with 10,000 ratings, the average is much more stable.
    3. Regression to the Mean: Most genres start with high averages because the first people to rate a "Documentary" or "Western" in a new system are usually fans
       of that genre. As the dataset grows, more casual viewers rate those movies, typically pulling the average down toward the global mean (usually around 3.5).

To address the decline in genre popularity fairly, we need to distinguish between a genuine drop in quality/interest and statistical noise caused by low sample
sizes in the early years.

The most professional way to handle this is using a Bayesian Average (similar to how IMDb calculates Top 250 rankings). This "pulls" the average of years with
few ratings toward the global mean, preventing them from having extreme, unearned peaks or valleys.

In [3]:
adjusted_results = analyzer.get_adjusted_trends(confidence_quantile=0.1)

print("--- Comparison: Raw vs. Adjusted Decrease / Top 5 (descending) ---")
comparison = adjusted_results[['genre', 'raw_decrease', 'adjusted_decrease', 'start_count']].sort_values(by='adjusted_decrease',ascending=False).head(5)
display(comparison)

print("--- Comparison: Raw vs. Adjusted Decrease / Top 15 (unsorted) ---")
comparison = adjusted_results[['genre', 'raw_decrease', 'adjusted_decrease', 'start_count']].head(15)
display(comparison)


--- Comparison: Raw vs. Adjusted Decrease / Top 5 (descending) ---


,genre,raw_decrease,adjusted_decrease,start_count
18,War,0.219478,0.332954,59973
7,Documentary,0.185136,0.264646,4736
5,Comedy,-0.324105,0.159630,2
15,Romance,0.174112,0.155119,231765
3,Animation,0.116184,0.141501,56977


--- Comparison: Raw vs. Adjusted Decrease / Top 15 (unsorted) ---


,genre,raw_decrease,adjusted_decrease,start_count
18,War,0.219478,0.332954,59973
7,Documentary,0.185136,0.264646,4736
5,Comedy,-0.324105,0.159630,2
15,Romance,0.174112,0.155119,231765
3,Animation,0.116184,0.141501,56977
13,Musical,0.165284,0.135371,55378
11,Horror,1.689543,0.111569,1
4,Children,0.147742,0.095651,102193
10,Film-Noir,-0.183983,0.081216,3446
1,Action,-0.434257,0.062439,1


Commentary on Step 3:
The raw analysis in Step 2 is highly sensitive to the 'Early Year' effect. For example, if a genre had only 50 ratings in its first year (1995), a few
5-star reviews from enthusiasts would inflate the average.

By applying a Bayesian adjustment:
1. We pull 'uncertain' early averages toward the global mean (~3.5).
2. Genres that showed a massive decrease simply because they started with
   a tiny, biased sample size will see their 'adjusted_decrease' shrink.
3. The resulting 'Top 5' is now much more representative of actual
   shifts in broad audience sentiment rather than statistical outliers.

#### 4.
Split the ratings into:

• a. A train dataset containing ratings before 2008

• b. A test dataset containing ratings for 2008 and 2009

• c. Apply a recommendation system using gradient descent

• d. Using the recommendation system, estimate a rating for the users and movies in the test dataset and compute the mean squared error

    Mapping IDs:
    Before splitting, we must map userId and movieId to contiguous integers (starting from 0). In the raw dataset, IDs might skip numbers
    (e.g., movie ID 500 followed by 505). For Matrix Factorization, we need these to correspond to row and column indices in our latent matrices.

    For step d. we create a new column in our test set so we can compare the Actual Rating vs. the Predicted Rating.

In [4]:
from src.recommender import MatrixFactorizer
from sklearn.metrics import mean_squared_error

# Initialize the model
model = MatrixFactorizer(n_factors=10, learning_rate=0.005, reg=0.02, epochs=3)

# a & b: Split
train, test = model.split_data(df)

# c: Fit using Gradient Descent
# WARNING: On 10M rows, this will take significant time.
# print("Training the model...")
# model.fit(train)

# To get decent results without crushing, train on 1m rows:
print("Training the model...")
model.fit(train.head(1000000))

# d: Estimate ratings for the test dataset (Evaluate)
# This creates the actual estimates
test_predictions = model.predict_test_set(test)

# --- Diagnostic Check ---
nan_count = test_predictions['predicted_rating'].isna().sum()
if nan_count > 0:
    print(f"⚠️ Warning: Found {nan_count} NaN predictions. Dropping them for MSE calculation.")

# --- Step 4d: Compute Mean Squared Error (Safe Version) ---
# We drop NaNs to ensure the MSE function doesn't crash
clean_predictions = test_predictions.dropna(subset=['rating', 'predicted_rating'])

mse = mean_squared_error(clean_predictions['rating'], clean_predictions['predicted_rating'])

print("\n--- Step 4d: Estimated Ratings (Actual vs Predicted) ---")
display(clean_predictions[['userId', 'movieId', 'title', 'rating', 'predicted_rating']].head(10))

print(f"\nFinal Mean Squared Error on Test Set: {mse:.4f}")
print(f"Evaluated on {len(clean_predictions)} samples.")

# # Display the estimates for the first 10 rows of the test set
# print("\n--- Step 4d: Estimated Ratings (Actual vs Predicted) ---")
# display(test_predictions[['userId', 'movieId', 'title', 'rating', 'predicted_rating']].head(10))
#
# # d: Compute Mean Squared Error
# mse = mean_squared_error(test_predictions['rating'], test_predictions['predicted_rating'])
# print(f"\nFinal Mean Squared Error on Test Set: {mse:.4f}")

Splitting data into train (<2008) and test (2008-2009)...
Training the model...
Training on 1000000 rows (Mean: 3.52)...
Epoch 1/3 complete.
Epoch 2/3 complete.
Epoch 3/3 complete.
Generating predictions for the test set...

--- Step 4d: Estimated Ratings (Actual vs Predicted) ---


,userId,movieId,title,rating,predicted_rating
4221,38,587,Ghost (1990),3.0,3.503015
4227,38,648,Mission: Impossible (1996),3.0,3.533356
4258,38,1721,Titanic (1997),3.0,3.543950
4307,38,3565,Where the Heart Is (2000),4.5,3.526682
4349,38,5349,Spider-Man (2002),4.0,3.556861
4363,38,6266,What a Girl Wants (2003),3.0,3.501302
4431,38,48385,Borat: Cultural Learnings of America for Make ...,0.5,3.565052
4432,38,48394,Pan's Labyrinth (El Laberinto del fauno) (2006),5.0,3.514716
4433,38,49272,Casino Royale (2006),2.0,3.532162
4434,38,54286,"Bourne Ultimatum, The (2007)",5.0,3.494963



Final Mean Squared Error on Test Set: 0.9973
Evaluated on 788225 samples.


#### 5.
Suppose you need to recommend 10 movies to a new user — what would you do? Which
movies would you recommend?

     The "Cold Start" Problem (New User)
     When a user is brand new to a platform, we face the Cold Start problem. Since we have no historical ratings for
     this user, our Gradient Descent model (Matrix Factorization) cannot calculate a unique user vector ($P_u$).

    In a professional setting, the standard approach is Popularity-Based Filtering. We recommend movies that are
    "globally" loved. However, we shouldn't just pick the highest average rating (a movie with one 5-star rating isn't
    necessarily better than The Godfather). We should recommend movies that have a high average rating and a large number
    of ratings (high confidence).

In [5]:
from recommender import RecommenderEngine

engine = RecommenderEngine(model, df)

print("\n--- New User Recommendations ---")
display(engine.recommend_cold_start(n=10))


--- New User Recommendations ---


,movieId,title,genres
24,260,Star Wars: Episode IV - A New Hope (a.k.a. Sta...,Action|Adventure|Sci-Fi
35,858,"Godfather, The (1972)",Crime|Drama
151,527,Schindler's List (1993),Drama|War
168,912,Casablanca (1942),Drama|Romance
189,1221,"Godfather: Part II, The (1974)",Crime|Drama
205,1193,One Flew Over the Cuckoo's Nest (1975),Comedy|Drama
208,1198,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure
243,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
1305,750,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War
1812,318,"Shawshank Redemption, The (1994)",Drama


#### 6.
Suppose we asked our new user to name the three most recent movies they enjoyed the
most, and they answered "Iron Man," "300," and "Transformers." Which 10 movies would you
recommend and why?

In [6]:
print("\n--- Targeted Recommendations (Iron Man, 300, Transformers) ---")
liked_movies = ["Iron Man", "300", "Transformers"]
display(engine.recommend_by_context(liked_movies, n=10))


--- Targeted Recommendations (Iron Man, 300, Transformers) ---


,movieId,title,genres
641,2100,Splash (1984),Comedy|Fantasy|Romance
672,2375,"Money Pit, The (1986)",Comedy
1928,1438,Dante's Peak (1997),Action|Thriller
3053,1003,Extreme Measures (1996),Drama|Thriller
17194,3669,Stay Tuned (1992),Comedy
101893,5083,Rare Birds (2001),Comedy|Drama
153055,6749,"Prince and the Pauper, The (1937)",Adventure|Drama
173373,4668,Some Girls (a.k.a. Sisters) (1988),Comedy|Drama|Romance
236725,7261,Barbershop 2: Back in Business (2004),Comedy
741705,6030,Girls! Girls! Girls! (1962),Comedy|Musical
